# Short-Volshock Action-Cap Stress Test v1

Goal: stress-test the best recent `short_volshock` action-cap overlay before converting it into a reward-timing objective or opening another PPO branch.

External basis:

- Qlib: replayable experiment workflow and backtest/report discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL: train/test/trade separation and transaction-cost-aware DRL framing: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Backtest overfitting risk from repeated candidate search: https://doi.org/10.21314/JCF.2016.322
- Volatility-managed portfolios: volatility state should shape exposure, but robustness margins matter before promotion: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431

This notebook does not retrain and does not promote. It applies stress haircuts to already-generated official warm-start replay metrics and audits whether the near-miss has any practical safety margin.


In [1]:

from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.official_candidate_aggregation import (
    OfficialReplayCandidateConfig,
    normalise_official_replay_candidate_frame,
    score_official_replay_candidate_pool,
    score_official_replay_stress_scenarios,
)

EXPERIMENT = "short_volshock_action_cap_stress_test_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / "runtime/model_selection/card/short_volshock_sell_guard_official_replay_v1"
MECHANISM_PK_CARD = ROOT / "runtime/model_selection/card/official_replay_candidate_mechanism_pk_v1"
CONFIG = OfficialReplayCandidateConfig(
    target_relative_return_pp=30.0,
    target_max_dd_pct=15.0,
    robust_relative_margin_pp=2.0,
    robust_max_dd_margin_pct=1.0,
    near_miss_relative_gap_pp=1.0,
    near_miss_max_dd_gap_pct=0.5,
)
STRESS_SCENARIOS = [
    {"scenario": "base_replay", "relative_haircut_pp": 0.0, "max_dd_add_pct": 0.0},
    {"scenario": "tiny_cost_buffer", "relative_haircut_pp": 0.25, "max_dd_add_pct": 0.10},
    {"scenario": "mild_execution_buffer", "relative_haircut_pp": 0.50, "max_dd_add_pct": 0.25},
    {"scenario": "strict_execution_buffer", "relative_haircut_pp": 1.00, "max_dd_add_pct": 0.50},
]
EXTERNAL_SOURCES = [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431",
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if hasattr(value, "item"):
        try:
            return json_safe(value.item())
        except Exception:
            pass
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")

source_selection = json.loads((SOURCE_CARD / "selection.json").read_text(encoding="utf-8"))
mechanism_selection = json.loads((MECHANISM_PK_CARD / "selection.json").read_text(encoding="utf-8"))
print({
    "experiment": EXPERIMENT,
    "source_decision": source_selection.get("decision"),
    "mechanism_decision": mechanism_selection.get("decision"),
    "mechanism_selected": mechanism_selection.get("selected_candidate"),
})


{'experiment': 'short_volshock_action_cap_stress_test_v1', 'source_decision': 'official_short_volshock_sell_guard_no_improvement_no_promotion', 'mechanism_decision': 'near_miss_action_cap_overlay_is_best_diagnostic_no_promotion', 'mechanism_selected': 'short_volshock_sell_guard::short_volshock_cap05_hit2_no_guard'}


In [2]:

raw = pd.read_csv(SOURCE_CARD / "official_replay_metrics.csv")
raw = raw[raw["status"].astype(str).eq("evaluated")].copy()
pool = normalise_official_replay_candidate_frame(
    raw,
    source_label="short_volshock_sell_guard",
    mechanism="action_cap_overlay",
    candidate_col="variant",
    config=CONFIG,
)
base_scores = score_official_replay_candidate_pool(pool, config=CONFIG)
stress_scores = score_official_replay_stress_scenarios(pool, stress_scenarios=STRESS_SCENARIOS, config=CONFIG)
trace_summary = pd.read_csv(SOURCE_CARD / "decision_trace_summary.csv")
selected_trace = pd.read_csv(SOURCE_CARD / "selected_decision_trace_summary.csv")

pool.to_csv(CARD / "action_cap_candidate_pool.csv", index=False)
base_scores.to_csv(CARD / "action_cap_base_scores.csv", index=False)
stress_scores.to_csv(CARD / "action_cap_stress_scores.csv", index=False)
trace_summary.to_csv(CARD / "source_decision_trace_summary.csv", index=False)
selected_trace.to_csv(CARD / "source_selected_decision_trace_summary.csv", index=False)
pd.DataFrame(STRESS_SCENARIOS).to_csv(CARD / "stress_scenarios.csv", index=False)

display(base_scores[["candidate_id", "full_gate", "robust_full_gate", "near_miss", "total_gate_loss", "min_relative_return", "max_max_dd", "failure_profile"]])
display(stress_scores[["scenario", "candidate_id", "stress_full_gate", "stress_gate_pass_count", "stress_total_gate_loss", "stress_min_relative_return", "stress_max_dd", "stress_failure_profile"]])
display(trace_summary[["variant", "fold", "overlay_gate_ready_days", "cap_changed_days", "cap_order_days", "blocked_by_min_rebalance_days", "blocked_by_trade_lock_days", "sell_order_days"]])


,candidate_id,full_gate,robust_full_gate,near_miss,total_gate_loss,min_relative_return,max_max_dd,failure_profile
0,short_volshock_sell_guard::short_volshock_cap0...,False,False,True,1.256545,29.817263,15.107381,near_miss
1,short_volshock_sell_guard::short_volshock_cap0...,False,False,False,5.155312,32.213744,15.515531,drawdown_control
2,short_volshock_sell_guard::short_volshock_cap0...,False,False,False,5.179090,32.320004,15.517909,drawdown_control
3,short_volshock_sell_guard::baseline_no_overlay,False,False,False,167.488279,30.262521,30.626799,drawdown_control


,scenario,candidate_id,stress_full_gate,stress_gate_pass_count,stress_total_gate_loss,stress_min_relative_return,stress_max_dd,stress_failure_profile
0,base_replay,short_volshock_sell_guard::short_volshock_cap0...,False,1,1.256545,29.817263,15.107381,stress_near_miss
1,base_replay,short_volshock_sell_guard::short_volshock_cap0...,False,1,5.155312,32.213744,15.515531,stress_near_miss
2,base_replay,short_volshock_sell_guard::short_volshock_cap0...,False,1,5.179090,32.320004,15.517909,stress_near_miss
3,base_replay,short_volshock_sell_guard::baseline_no_overlay,False,0,167.488279,30.262521,30.626799,stress_drawdown_control
4,mild_execution_buffer,short_volshock_sell_guard::short_volshock_cap0...,False,1,4.256545,29.317263,15.357381,stress_near_miss
5,mild_execution_buffer,short_volshock_sell_guard::short_volshock_cap0...,False,1,7.655312,31.713744,15.765531,stress_near_miss
6,mild_execution_buffer,short_volshock_sell_guard::short_volshock_cap0...,False,1,7.679090,31.820004,15.767909,stress_near_miss
7,mild_execution_buffer,short_volshock_sell_guard::baseline_no_overlay,False,0,172.725759,29.762521,30.876799,stress_drawdown_control
8,strict_execution_buffer,short_volshock_sell_guard::short_volshock_cap0...,False,1,7.256545,28.817263,15.607381,stress_near_miss
9,strict_execution_buffer,short_volshock_sell_guard::short_volshock_cap0...,False,1,10.155312,31.213744,16.015531,stress_near_miss


,variant,fold,overlay_gate_ready_days,cap_changed_days,cap_order_days,blocked_by_min_rebalance_days,blocked_by_trade_lock_days,sell_order_days
0,baseline_no_overlay,wf_2022,0,0,0,267,117,17
1,baseline_no_overlay,wf_2023_2024,0,0,0,260,126,24
2,short_volshock_cap05_hit2_no_guard,wf_2022,236,236,17,279,50,13
3,short_volshock_cap05_hit2_no_guard,wf_2023_2024,64,58,8,260,113,25
4,short_volshock_cap05_hit2_sell_guard_1pct,wf_2022,236,236,16,279,28,14
5,short_volshock_cap05_hit2_sell_guard_1pct,wf_2023_2024,64,58,8,257,109,27
6,short_volshock_cap05_hit2_sell_guard_zero,wf_2022,236,236,39,256,27,35
7,short_volshock_cap05_hit2_sell_guard_zero,wf_2023_2024,64,58,11,254,109,30


In [3]:

scenario_summary_rows = []
for scenario, group in stress_scores.groupby("scenario", sort=False):
    scenario_summary_rows.append(
        {
            "scenario": scenario,
            "candidate_count": int(group["candidate_id"].nunique()),
            "full_gate_count": int(group["stress_full_gate"].sum()),
            "best_candidate_id": str(group.sort_values(["stress_full_gate", "stress_total_gate_loss"], ascending=[False, True]).iloc[0]["candidate_id"]),
            "best_loss": float(group["stress_total_gate_loss"].min()),
            "best_min_relative_return": float(group.sort_values("stress_total_gate_loss").iloc[0]["stress_min_relative_return"]),
            "best_max_dd": float(group.sort_values("stress_total_gate_loss").iloc[0]["stress_max_dd"]),
        }
    )
scenario_summary = pd.DataFrame(scenario_summary_rows)
scenario_summary.to_csv(CARD / "stress_scenario_summary.csv", index=False)

selected_candidate_id = str(base_scores.iloc[0]["candidate_id"]) if not base_scores.empty else ""
selected_stress = stress_scores[stress_scores["candidate_id"].eq(selected_candidate_id)].copy()
selected_stress.to_csv(CARD / "selected_candidate_stress_profile.csv", index=False)
selected_trace_rows = trace_summary[trace_summary["variant"].eq(selected_candidate_id.split("::", 1)[-1])].copy()
selected_trace_rows.to_csv(CARD / "selected_candidate_trace_absorption.csv", index=False)

base_top = base_scores.iloc[0].to_dict() if not base_scores.empty else {}
base_full_gate = bool(base_top.get("full_gate", False))
base_near_miss = bool(base_top.get("near_miss", False))
stress_pass_count = int(selected_stress["stress_full_gate"].sum()) if not selected_stress.empty else 0
stress_scenario_count = int(selected_stress["scenario"].nunique()) if not selected_stress.empty else 0
best_base_loss = float(base_top.get("total_gate_loss", math.nan)) if base_top else math.nan
min_rel = float(base_top.get("min_relative_return", math.nan)) if base_top else math.nan
max_dd = float(base_top.get("max_max_dd", math.nan)) if base_top else math.nan
cap_order_days = int(selected_trace_rows["cap_order_days"].sum()) if not selected_trace_rows.empty else 0
cap_changed_days = int(selected_trace_rows["cap_changed_days"].sum()) if not selected_trace_rows.empty else 0
blocked_by_min_rebalance = int(selected_trace_rows["blocked_by_min_rebalance_days"].sum()) if not selected_trace_rows.empty else 0

if base_full_gate and stress_pass_count == stress_scenario_count:
    decision = "short_volshock_action_cap_stress_robust_no_promotion"
    next_mechanism = "run fixed-split plus wf_2020 official replay before retaining action-cap behavior"
elif base_full_gate:
    decision = "short_volshock_action_cap_low_margin_full_gate_no_promotion"
    next_mechanism = "convert action-cap behavior into reward-timing objective with explicit margin target"
elif base_near_miss:
    decision = "short_volshock_action_cap_near_miss_stress_fragile_no_promotion"
    next_mechanism = "reward-timing objective should target at least +0.5pp excess and -0.25pct MaxDD margin rather than copying the overlay threshold"
else:
    decision = "short_volshock_action_cap_stress_failed_no_promotion"
    next_mechanism = "drop action-cap execution overlay as a promotion path and derive a new training objective handoff"

conclusion = [
    f"Selected candidate {selected_candidate_id}: base full_gate={base_full_gate}, near_miss={base_near_miss}, min_rel={min_rel:.2f}pp, max_dd={max_dd:.2f}%, loss={best_base_loss:.2f}.",
    f"Stress pass count for selected candidate: {stress_pass_count}/{stress_scenario_count}; no promotion unless every stress scenario passes.",
    f"Trace absorption: cap_changed_days={cap_changed_days}, cap_order_days={cap_order_days}, blocked_by_min_rebalance_days={blocked_by_min_rebalance}.",
    f"Next mechanism: {next_mechanism}.",
]

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "direct_comparison_baseline": "short_volshock_sell_guard_official_replay_v1",
    "changed_layer": ["model_selection", "execution", "stress_test"],
    "external_reference": EXTERNAL_SOURCES,
    "hypothesis": "The near-miss action-cap overlay may be too fragile to promote; if it cannot survive small official-metric stress buffers, it should be converted into a reward-timing target with margin instead of tuned as an execution threshold.",
    "acceptance_criteria": {
        "base_gate": ">=30pp relative and <=15% MaxDD in every supplied official replay window",
        "stress_gate": "same gate after 0.25-1.0pp relative haircut and 0.10-0.50pct MaxDD add-on scenarios",
    },
    "accepted_tradeoff": "No promotion from near-miss or stress-fragile behavior; use only as diagnostic reward-timing signal.",
    "failure_mode": "Threshold search can overfit the two official replay windows when the margin is below plausible execution/cost noise.",
    "closest_historical_success_case": "short_volshock_sell_guard_official_replay_v1 found cap05_hit2_no_guard near the official gate with active cap-order behavior.",
    "closest_historical_failure_case": "official_replay_candidate_mechanism_pk_v1 showed no robust official replay candidate and hard target caps failed trend capture.",
    "duplicate_check": "This run does not add a new overlay threshold; it stress-tests existing official replay candidates and decision traces.",
    "selected_candidate": selected_candidate_id,
    "selected_base_score": base_top,
    "stress_pass_count": stress_pass_count,
    "stress_scenario_count": stress_scenario_count,
    "promotion_allowed": False,
    "next_mechanism": next_mechanism,
    "sidecars": [
        "action_cap_candidate_pool.csv",
        "action_cap_base_scores.csv",
        "action_cap_stress_scores.csv",
        "stress_scenarios.csv",
        "stress_scenario_summary.csv",
        "selected_candidate_stress_profile.csv",
        "source_decision_trace_summary.csv",
        "source_selected_decision_trace_summary.csv",
        "selected_candidate_trace_absorption.csv",
        "selection.json",
        "summary.json",
    ],
}
summary = {
    "experiment": EXPERIMENT,
    "created_at": selection["created_at"],
    "decision": decision,
    "promotion_allowed": False,
    "base_scores": base_scores.to_dict("records"),
    "stress_scores": stress_scores.to_dict("records"),
    "scenario_summary": scenario_summary.to_dict("records"),
    "selected_stress_profile": selected_stress.to_dict("records"),
    "trace_absorption": selected_trace_rows.to_dict("records"),
    "conclusion": conclusion,
    "next_mechanism": next_mechanism,
    "external_sources": EXTERNAL_SOURCES,
    "sidecars": selection["sidecars"],
}
write_json(CARD / "selection.json", selection)
write_json(CARD / "summary.json", summary)

display(scenario_summary)
display(selected_stress[["scenario", "stress_full_gate", "stress_gate_pass_count", "stress_total_gate_loss", "stress_min_relative_return", "stress_max_dd", "stress_failure_profile"]])
display(Markdown("## Runtime Conclusion\n" + "\n".join(f"- {item}" for item in conclusion)))
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))


,scenario,candidate_count,full_gate_count,best_candidate_id,best_loss,best_min_relative_return,best_max_dd
0,base_replay,4,0,short_volshock_sell_guard::short_volshock_cap0...,1.256545,29.817263,15.107381
1,mild_execution_buffer,4,0,short_volshock_sell_guard::short_volshock_cap0...,4.256545,29.317263,15.357381
2,strict_execution_buffer,4,0,short_volshock_sell_guard::short_volshock_cap0...,7.256545,28.817263,15.607381
3,tiny_cost_buffer,4,0,short_volshock_sell_guard::short_volshock_cap0...,2.506545,29.567263,15.207381


,scenario,stress_full_gate,stress_gate_pass_count,stress_total_gate_loss,stress_min_relative_return,stress_max_dd,stress_failure_profile
0,base_replay,False,1,1.256545,29.817263,15.107381,stress_near_miss
4,mild_execution_buffer,False,1,4.256545,29.317263,15.357381,stress_near_miss
8,strict_execution_buffer,False,1,7.256545,28.817263,15.607381,stress_near_miss
12,tiny_cost_buffer,False,1,2.506545,29.567263,15.207381,stress_near_miss


## Runtime Conclusion
- Selected candidate short_volshock_sell_guard::short_volshock_cap05_hit2_no_guard: base full_gate=False, near_miss=True, min_rel=29.82pp, max_dd=15.11%, loss=1.26.
- Stress pass count for selected candidate: 0/4; no promotion unless every stress scenario passes.
- Trace absorption: cap_changed_days=294, cap_order_days=25, blocked_by_min_rebalance_days=539.
- Next mechanism: reward-timing objective should target at least +0.5pp excess and -0.25pct MaxDD margin rather than copying the overlay threshold.

{
  "experiment": "short_volshock_action_cap_stress_test_v1",
  "created_at": "2026-06-01T03:02:41.383992+00:00",
  "decision": "short_volshock_action_cap_near_miss_stress_fragile_no_promotion",
  "role": "diagnose",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "short_volshock_sell_guard_official_replay_v1",
  "changed_layer": [
    "model_selection",
    "execution",
    "stress_test"
  ],
  "external_reference": [
    "https://github.com/microsoft/qlib",
    "https://arxiv.org/abs/2009.11189",
    "https://github.com/AI4Finance-Foundation/FinRL",
    "https://arxiv.org/abs/2011.09607",
    "https://doi.org/10.21314/JCF.2016.322",
    "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2659431"
  ],
  "hypothesis": "The near-miss action-cap overlay may be too fragile to promote; if it cannot survive small official-metric stress buffers, it should be converted into a reward-timing target with margin instead of tuned as an execution t

## Runtime Conclusion

The executable conclusion is produced by the previous cell and persisted to `selection.json` / `summary.json`. Keep this card as diagnostic evidence only unless a later replay proves robust full-gate behavior.
